In [1]:
from scipy.stats import norm, bootstrap
import statsmodels.formula.api as smf
import statsmodels.api as sm
import numpy as np

from joblib import Parallel, delayed

from p2prev import PCurveMixture, PCurveWithinGroupDifference
from p2prev.model import p_curve_loglik
from util.pybridgesampling import bridge_sample
import pymc as pm
import arviz as az

from bids import BIDSLayout
layout = BIDSLayout('bids_dataset')

In [2]:
def calc_dprime(x_true, x_resp):
    '''
    calculates d-prime (d')
    '''
    hit_rate = x_resp[x_true].mean()
    fa_rate = x_resp[~x_true].mean()
    d_prime = norm.ppf(hit_rate) - norm.ppf(fa_rate)
    return d_prime

In [3]:
participants = layout.get(suffix = 'participants', extension = 'tsv')[0].get_df()
high_imp = participants.notes.str.contains('exceeded') == True
sub_nums = participants.participant_id.str.extract(r'sub-(\w+)')
good_subs = sub_nums[~high_imp].iloc[:, 0].to_list()
print('Excluded', sub_nums[high_imp][0].to_list())
print('%d subjects remaining.'%len(good_subs))

ps_disc = []
ps_stream = []
dprimes = []

for sub in good_subs:

    # load behavioral data
    discrimination, streaming = layout.get(subject = sub, suffix = 'events', extension = 'tsv')
    discrimination, streaming = discrimination.get_df(), streaming.get_df()
    # make sure we're counting each trial once, not each tone
    discrimination = discrimination[discrimination.tone_num == 1].reset_index(drop = True)
    streaming = streaming[streaming.tone_num == 1].reset_index(drop = True)
    
    # heartbeat discrimination task
    d = calc_dprime(discrimination.trial_type == 'systole', discrimination.response == 'systole')
    discrimination['systole'] = discrimination.response == 'systole'
    logit = smf.glm('systole ~ trial_type', data = discrimination, family = sm.families.Binomial()).fit()
    p_disc = logit.pvalues['trial_type[T.systole]']
    ps_disc.append(p_disc)
    dprimes.append(d)
    
    streaming['streaming'] = streaming.response == 'streaming'
    logit = smf.glm('streaming ~ trial_type', data = streaming, family = sm.families.Binomial()).fit()
    p_stream = logit.pvalues['trial_type[T.systole]']
    ps_stream.append(p_stream)

ps_disc = np.array(ps_disc)
ps_stream = np.array(ps_stream)
ps_disc[ps_disc == 1] -= 1e-6 # mixture model can't deal with p=1 exactly
ps_stream[ps_stream == 1] -= 1e-6 

Excluded ['08', '19']
20 subjects remaining.


In [4]:
res = bootstrap([dprimes], np.mean, confidence_level = .95, n_resamples = 10000, random_state = 0)
res.bootstrap_distribution.mean(), res.confidence_interval

(0.26785943271621643,
 ConfidenceInterval(low=0.12199999991537479, high=0.49979744680674976))

In [5]:
mod = PCurveWithinGroupDifference(ps_stream, ps_disc, idata_kwargs = dict(include_transformed = True))
mod.fit()
logml_full = bridge_sample(mod._model, mod._mix, random_seed = 0)
mod.summary(hdi_prob = .95)

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (5 chains in 5 jobs)
NUTS: [effect_size, prevalence]


Output()

Sampling 5 chains for 1_000 tune and 1_000 draw iterations (5_000 + 5_000 draws total) took 9 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
effect_size[0],3.069,0.856,1.542,4.898,0.020,0.014,1951.0,1050.0,1.0
effect_size[1],2.527,0.724,1.065,3.918,0.013,0.009,3139.0,2240.0,1.0
effect_size_diff,-0.542,1.120,-2.872,1.499,0.023,0.016,2404.0,1641.0,1.0
effect_size_log__[0],1.072,0.345,0.482,1.630,0.012,0.008,1951.0,1050.0,1.0
effect_size_log__[1],0.879,0.328,0.212,1.435,0.007,0.005,3139.0,2240.0,1.0
prevalence[0],0.549,0.129,0.298,0.787,0.002,0.002,3675.0,2924.0,1.0
prevalence[1],0.158,0.088,0.010,0.326,0.002,0.001,3126.0,2278.0,1.0
prevalence[2],0.229,0.117,0.022,0.450,0.002,0.002,2513.0,2207.0,1.0
prevalence[3],0.063,0.062,0.000,0.186,0.001,0.001,2494.0,2431.0,1.0
prevalence_H1,0.222,0.103,0.047,0.415,0.002,0.002,3052.0,2162.0,1.0


In [6]:
# calculate posteriors for conditional prevalences 
prev = mod.mixture.posterior.prevalence
k00 = prev[...,0].values.flatten()
k10 = prev[...,1].values.flatten()
k01 = prev[...,2].values.flatten()
k11 = prev[...,3].values.flatten()

prev2 = k01 + k11
prob_H1_given_H0 = k10 / (1 - prev2)
prob_H1_given_H2 = k11 / prev2

print(prob_H1_given_H0.mean(), az.hdi(prob_H1_given_H0, hdi_prob = .95))
print(prob_H1_given_H2.mean(), az.hdi(prob_H1_given_H2, hdi_prob = .95))

0.2247168703099096 [0.01250991 0.45043779]
0.22443129464964276 [9.18898444e-05 6.32538321e-01]


In [7]:
with pm.Model() as nested_model:
    '''
    this is just model definition from p2prev.PCurveWithinGroupDifference
    with the H1=True & H2=False group removed from the mixture 
    '''

    # define likelihoods of p-values under H0, H1, and H2
    delta = pm.Exponential('effect_size', lam = 1/mod.prior, shape = 2)
    pm.Deterministic('effect_size_diff', delta[1] - delta[0])
    pcurve_H1 = pm.CustomDist.dist(delta[0], logp = p_curve_loglik)
    pcurve_H2 = pm.CustomDist.dist(delta[1], logp = p_curve_loglik)
    pcurve_H0 = pm.Uniform.dist(0, 1)

    # probabilities of combinations of H0, H1, and H2 being true
    k = pm.Dirichlet('prevalence', np.ones(3)) # prior
    k00, k01, k11 = k[0], k[1], k[2] # notation from Ince
    prev1 = pm.Deterministic('prevalence_H1', k11)
    prev2 = pm.Deterministic('prevalence_H2', k01 + k11)
    pm.Deterministic('prevalence_diff', prev2 - prev1)
    pm.Deterministic('prob_H2_given_H1', k11 / prev1)
    pm.Deterministic('prob_H2_given_H0', k01 / (1 - prev1))
    pm.Deterministic('prob_H1_given_H2', k11 / prev2)

    # define likelihoods for combinations of H0, H1, and H2 being true
    H1_logl_1 = pm.logp(pcurve_H1, mod._ps1)
    H0_logl_1 = pm.logp(pcurve_H0, mod._ps1)
    H2_logl_2 = pm.logp(pcurve_H2, mod._ps2)
    H0_logl_2 = pm.logp(pcurve_H0, mod._ps2)
    logl_00 = H0_logl_1 + H0_logl_2
    logl_01 = H0_logl_1 + H2_logl_2
    logl_11 = H1_logl_1 + H2_logl_2

    # and marginalize these likelihoods over class probabilities
    logp_marg = pm.logaddexp(
        pm.math.log(k00) + logl_00,
        pm.math.log(k01) + logl_01,
        pm.math.log(k11) + logl_11
    )
    # logp_marg isn't technically a valid PyMC likelihood
    # (though it's a real log-likelihood), but it will be treated
    # as one during sampling if we specify it as a "potential"
    pm.Potential('likelihood', logp_marg)

    # so now we can sample from posterior
    nested_idata = pm.sample(**mod.sampler_kwargs)

logml_nested = bridge_sample(nested_model, nested_idata, random_seed = 0)

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (5 chains in 5 jobs)
NUTS: [effect_size, prevalence]


Output()

Sampling 5 chains for 1_000 tune and 1_000 draw iterations (5_000 + 5_000 draws total) took 9 seconds.


In [8]:
# Bayes Factor to asses whether there are people who 
# show the streaming effect but not interoceptive awareness
bf = np.exp(logml_full['logml'] - logml_nested['logml'])
print('Bayes Factor, full vs. nested model:', bf)

Bayes Factor, full vs. nested model: 8.75488740820014
